<center><h1>Physics Informed Neural Networks</h1></center>
<center><h2>Project</h2></center>
<center><h3>Harsh Solanki</h3></center>
<center><h4>harsh.solanki@student.uni-tuebingen.de</h4></center>

IMP references: PINN_05 and PINN_06

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import scipy as sp
from IPython import display
import os, sys, re
import numpy as np
import matplotlib.pyplot as plt 
from scipy.interpolate import interp1d

In [8]:
def eos(P, K, gamma):
    if P <= 0:
        return 0.0, 0.0
    rho_b = (P/K)**(1/gamma)
    epsilon = rho_b + P/(gamma - 1)
    return rho_b, epsilon

def solve_tov_full(rho_c=1.28e-3, K=100, gamma=2, dr=0.05, r_max=15):
    P_c = K * rho_c**gamma
    _, eps_c = eos(P_c, K, gamma)
    
    r_int, m_int, P_int = [0.0], [0.0], [P_c]
    
    r0 = 1e-6
    m0 = (4.0/3.0) * np.pi * eps_c * r0**3
    P0 = P_c - (2.0/3.0) * np.pi * (eps_c + P_c) * (eps_c + 3.0*P_c) * r0**2
    
    r, m, P = r0, m0, P0
    r_int.append(r); m_int.append(m); P_int.append(P)
    
    while P > 1e-12 and r < r_max:
        rho_b, epsilon = eos(P, K, gamma)
        dm_dr = 4 * np.pi * r**2 * epsilon
        denom = r * (r - 2 * m)
        denom = denom if abs(denom) > 1e-10 else 1e-10
        dP_dr = - (epsilon + P) * (m + 4 * np.pi * r**3 * P) / denom
        
        m += dr * dm_dr
        P = max(P + dr * dP_dr, 0.0)
        r += dr
        
        r_int.append(r); m_int.append(m); P_int.append(P)
    
    R = r_int[-1]
    M = m_int[-1]
    return M, R, np.array(r_int), np.array(m_int), np.array(P_int)

print("Generating TOV solution...")
M_true, R_true, r_data, m_data, P_data = solve_tov_full(
    rho_c=1.28e-3, K=100, gamma=2, dr=0.05, r_max=15
)
print("done!")

Generating TOV solution...
done!


In [23]:
mask = r_data > 1e-6
r_data, m_data, P_data = r_data[mask], m_data[mask], P_data[mask]

#print(P_data[187])

surface_idx = np.where(P_data < 1e-8)[0]
print(f"P[0]: {P_data[0]}")
R_surface = r_data[surface_idx[0]] if len(surface_idx) > 0 else r_data[-1]

print(f"Data points: {len(r_data)}, R_star: {R_true:.4f}, M_star: {M_true:.6f}")
print(f"R_min: {min(r_data)}, P_min: {min(P_data)}, M_min: {min(m_data)}")

P[0]: 0.00016383999999347693
Data points: 189, R_star: 9.4500, M_star: 1.403158
R_min: 0.050001000000000004, P_min: 0.0, M_min: 9.071974753346467e-16


Normalize the training data and convert into tensorflow tensor

<centre>min-max normalize:<centre>
$$2\frac{x - x_{min}}{x_{max} - x_{min}} -1$$

In [30]:
r_min, r_max = 0.0, R_surface
m_min, m_max = 0.0, M_true
P_min, P_max = 0.0, P_data[0]

def norm_r(r): return (r - r_min) / (r_max - r_min)# + 1e-10)
def norm_m(m): return (m - m_min) / (m_max - m_min)# + 1e-10)
def norm_P(P): return (P - P_min) / (P_max - P_min)# + 1e-10)

def denorm_r(r_n): return r_n * (r_max - r_min) + r_min
def denorm_m(m_n): return m_n * (m_max - m_min) + m_min
def denorm_P(P_n): return P_n * (P_max - P_min) + P_min

r_tf = tf.constant(norm_r(r_data).reshape(-1, 1), dtype=tf.float64)
m_tf = tf.constant(norm_m(m_data).reshape(-1, 1), dtype=tf.float64)
P_tf = tf.constant(norm_P(P_data).reshape(-1, 1), dtype=tf.float64)

print(f"---Shapes---")
print(f"r_tf: {r_tf.shape}, m_tf: {m_tf.shape}, p_tf: {P_tf.shape}")

norm_consts = {'r_min': r_min, 'r_max': r_max, 'm_min': m_min, 'm_max': m_max, 
               'P_min': P_min, 'P_max': P_max}

print(f"r_min = {min(r_tf)}, r_max = {max(r_tf)}")
print(f"m_min = {min(m_tf)}, m_max = {max(m_tf)}")
print(f"P_min = {min(P_tf)}, P_max = {max(P_tf)}")
# r_nan = np.isnan(r_tf).any()
# m_nan = np.isnan(m_tf).any()
# P_nan = np.isnan(P_tf).any()
# print(r_nan, m_nan, P_nan)


---Shapes---
r_tf: (189, 1), m_tf: (189, 1), p_tf: (189, 1)
r_min = [0.00531925], r_max = [1.00531915]
m_min = [6.46539903e-16], m_max = [1.]
P_min = [0.], P_max = [1.]


In [32]:
class PINN_TOV(tf.keras.Model):
    def __init__(self):
        super(PINN_TOV, self).__init__()
        self.hidden1 = tf.keras.layers.Dense(128, activation='tanh', dtype=tf.float64,
                                              kernel_initializer='glorot_uniform')
        self.hidden2 = tf.keras.layers.Dense(128, activation='tanh', dtype=tf.float64,
                                              kernel_initializer='glorot_uniform')
        self.hidden3 = tf.keras.layers.Dense(128, activation='tanh', dtype=tf.float64,
                                              kernel_initializer='glorot_uniform')
        self.out = tf.keras.layers.Dense(2, activation=None, dtype=tf.float64)
        
        # Physics parameters (separate from network weights)
        self.log_K = tf.Variable(np.log(50.0), dtype=tf.float64, name='log_K')
        self.log_rho_c = tf.Variable(np.log(4.8e-4), dtype=tf.float64, name='log_rho_c')
        self.gamma = tf.Variable(1.5, dtype=tf.float64, name='gamma')
    
    @property
    def nn_weights(self):
        """Get only neural network weights (not physics parameters)"""
        return (self.hidden1.trainable_variables + 
                self.hidden2.trainable_variables + 
                self.hidden3.trainable_variables + 
                self.out.trainable_variables)
    
    @property
    def physics_vars(self):
        """Get physics parameters"""
        return [self.log_K, self.log_rho_c, self.gamma]
    
    @property
    def all_trainable_vars(self):
        """Get all trainable variables"""
        return self.nn_weights + self.physics_vars
    
    def call(self, r):
        x = self.hidden1(r)
        x = self.hidden2(x)
        x = self.hidden3(x)
        return self.out(x)
    
    def get_physics_params(self):
        K = tf.exp(self.log_K)
        gamma = tf.clip_by_value(self.gamma, 1.1, 3.0)
        rho_c = tf.exp(self.log_rho_c)
        return K, gamma, rho_c

model = PINN_TOV()

# Verify variables are collected correctly
print(f"\nNeural network variables: {len(model.nn_weights)}")
print(f"Physics variables: {len(model.physics_vars)}")
print(f"Total trainable: {len(model.all_trainable_vars)}")



Neural network variables: 0
Physics variables: 3
Total trainable: 3


The total loss function consists of three components:

$$\mathcal{L}_{total} = \lambda_1 \mathcal{L}_{data} + \lambda_2 \mathcal{L}_{TOV} + \lambda_3 \mathcal{L}_{BC}$$

1. Data Loss $\mathcal{L}_{data}$

$$\mathcal{L}_{data} = \frac{1}{N}\sum_{i=1}^{N} \left(\hat{m}_{NN}(r_i) - \hat{m}_{data}(r_i)\right)^2 + \frac{1}{N}\sum_{i=1}^{N} \left(\hat{P}_{NN}(r_i) - \hat{P}_{data}(r_i)\right)^2$$

where $\hat{m}$ and $\hat{P}$ denote normalized quantities.

2. TOV Loss $\mathcal{L}_{TOV}$

$$\frac{dm}{dr} = 4\pi r^2 \varepsilon$$

$$\frac{dP}{dr} = -\frac{(\varepsilon + P)(m + 4\pi r^3 P)}{r(r - 2m)}$$

The residuals are:

$$\mathcal{R}_m = \frac{dm_{NN}}{dr} - 4\pi r^2 \varepsilon$$

$$\mathcal{R}_P = \frac{dP_{NN}}{dr} + \frac{(\varepsilon + P_{NN})(m_{NN} + 4\pi r^3 P_{NN})}{r(r - 2m_{NN})}$$

So the physics loss is:

$$\mathcal{L}_{TOV} = \frac{1}{N_c}\sum_{i=1}^{N_c} \mathcal{R}_m^2(r_i) + \frac{1}{N_c}\sum_{i=1}^{N_c} \mathcal{R}_P^2(r_i)$$

where the energy density $\varepsilon$ is computed from the polytropic equation of state:

$$\rho_b = \left(\frac{P}{K}\right)^{1/\gamma}, \qquad \varepsilon = \rho_b + \frac{P}{\gamma - 1}$$

and $N_c$ is the number of points.

3. Boundary Condition Loss $\mathcal{L}_{BC}$

Enforces the central boundary conditions at $r = r_0 \approx 0$:

$$\mathcal{L}_{BC} = \left(m_{NN}(r_0) - 0\right)^2 + \left(P_{NN}(r_0) - P_c\right)^2$$

where the central pressure is computed from the trainable parameters:

$$P_c = K \rho_c^{\gamma}$$

with $K = e^{\log K}$, $\rho_c = e^{\log \rho_c}$ recovered from log-space variables.

## Summary of Trainable Parameters

| Parameter | Stored As | Recovered Via |
|-----------|-----------|---------------|
| $K$ | $\log K$ | $e^{\log K}$ |
| $\gamma$ | $\gamma$ | directly |
| $\rho_c$ | $\log \rho_c$ | $e^{\log \rho_c}$ |

In [36]:
def compute_loss(model, r_tf, m_tf, P_tf, norm,
                 lam_data=1.0, lam_tov=1.0, lam_bc=1.0, lam_surf=1.0):

    pi  = tf.constant(np.pi, dtype=tf.float64)
    eps = tf.constant(1e-10, dtype=tf.float64)

    K     = tf.exp(model.log_K)
    gamma = tf.clip_by_value(model.gamma, 1.1, 3.0)
    rho_c = tf.exp(model.log_rho_c)

    dr_scale = norm['r_max'] - norm['r_min']
    dm_scale = norm['m_max'] - norm['m_min']
    dP_scale = norm['P_max'] - norm['P_min']

    def denorm_r(x): return x * dr_scale + norm['r_min']
    def denorm_m(x): return x * dm_scale + norm['m_min']
    def denorm_P(x): return x * dP_scale + norm['P_min']

    # ── DATA LOSS ──────────────────────────────────────────────────────────
    pred   = model(r_tf)
    m_pred = pred[:, 0:1]
    P_pred = pred[:, 1:2]

    loss_data = (tf.reduce_mean(tf.square(m_pred - m_tf)) +
                 tf.reduce_mean(tf.square(P_pred - P_tf)))

    # ── PHYSICS LOSS ───────────────────────────────────────────────────────
    with tf.GradientTape(persistent=True) as phys_tape:
        phys_tape.watch(r_tf)
        pred_col = model(r_tf)
        m_n = pred_col[:, 0:1]
        P_n = pred_col[:, 1:2]

    dm_dr = phys_tape.gradient(m_n, r_tf) * dm_scale / dr_scale
    dP_dr = phys_tape.gradient(P_n, r_tf) * dP_scale / dr_scale
    del phys_tape

    r_ph = denorm_r(r_tf)
    m_ph = denorm_m(m_n)
    P_ph = tf.maximum(denorm_P(P_n), eps)

    rho_b   = tf.maximum(P_ph / K, eps) ** (1.0 / gamma)
    epsilon = rho_b + P_ph / tf.maximum(gamma - 1.0, eps)

    denom = r_ph * (r_ph - 2.0 * m_ph)
    denom = tf.where(tf.abs(denom) < eps, eps * tf.ones_like(denom), denom)

    res_m = dm_dr - 4.0 * pi * r_ph**2 * epsilon
    res_P = dP_dr + (epsilon + P_ph) * (m_ph + 4.0 * pi * r_ph**3 * P_ph) / denom

    loss_tov = (tf.reduce_mean(tf.square(res_m)) +
                tf.reduce_mean(tf.square(res_P)))

    # ── BOUNDARY CONDITION at r = 0 ────────────────────────────────────────
    r0      = tf.constant([[0.0]], dtype=tf.float64)
    pred_bc = model(r0)
    m_bc    = denorm_m(pred_bc[:, 0:1])
    P_bc    = denorm_P(pred_bc[:, 1:2])
    P_c     = K * rho_c ** gamma

    loss_bc = (tf.reduce_mean(tf.square(m_bc)) +
               tf.reduce_mean(tf.square(P_bc - P_c)) / (P_c**2 + eps))

    # ── SURFACE CONDITION at r = R (normalized r = 1) ──────────────────────
    r_surf    = tf.constant([[1.0]], dtype=tf.float64)
    pred_surf = model(r_surf)
    P_surf    = denorm_P(pred_surf[:, 1:2])

    loss_surf = tf.reduce_mean(tf.square(P_surf)) / (dP_scale**2 + eps)

    # ── TOTAL ──────────────────────────────────────────────────────────────
    loss_total = (lam_data * loss_data +
                  lam_tov  * loss_tov  +
                  lam_bc   * loss_bc   +
                  lam_surf * loss_surf)

    return loss_total, loss_data, loss_tov, loss_bc, loss_surf

In [38]:
history = {'loss_total': [], 'loss_data': [], 'loss_tov': [],
           'loss_bc': [], 'loss_surf': [], 'K': [], 'gamma': [], 'rho_c': []}

optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)

for epoch in range(500):

    with tf.GradientTape() as tape:
        loss_total, loss_data, loss_tov, loss_bc, loss_surf = compute_loss(
            model, r_tf, m_tf, P_tf, norm_consts,
            lam_data=10.0, lam_tov=1.0, lam_bc=10.0, lam_surf=1.0
        )

    grads = tape.gradient(loss_total, model.trainable_variables)
    grads = [tf.clip_by_norm(g, 1.0) if g is not None else None for g in grads]
    optimizer.apply_gradients(zip(grads, model.trainable_variables))

    K_val     = tf.exp(model.log_K).numpy()
    gamma_val = tf.clip_by_value(model.gamma, 1.1, 3.0).numpy()
    rho_c_val = tf.exp(model.log_rho_c).numpy()

    history['loss_total'].append(loss_total.numpy())
    history['loss_data'].append(loss_data.numpy())
    history['loss_tov'].append(loss_tov.numpy())
    history['loss_bc'].append(loss_bc.numpy())
    history['loss_surf'].append(loss_surf.numpy())
    history['K'].append(K_val)
    history['gamma'].append(gamma_val)
    history['rho_c'].append(rho_c_val)

    if epoch % 50 == 0:
        print(f"[{epoch:5d}] total={loss_total.numpy():.3e} | "
              f"data={loss_data.numpy():.3e} | tov={loss_tov.numpy():.3e} | "
              f"bc={loss_bc.numpy():.3e} | surf={loss_surf.numpy():.3e} | "
              f"K={K_val:.3f} ({abs(K_val-100)/100*100:.1f}%) | "
              f"γ={gamma_val:.4f} ({abs(gamma_val-2.0)/2.0*100:.1f}%) | "
              f"ρ_c={rho_c_val:.3e} ({abs(rho_c_val-1.28e-3)/1.28e-3*100:.1f}%)")

[    0] total=2.041e+00 | data=1.081e-01 | tov=1.644e-02 | bc=9.404e-02 | surf=3.154e-03 | K=49.875 (50.1%) | γ=1.5025 (24.9%) | ρ_c=4.788e-04 (62.6%)
[   50] total=1.659e+00 | data=9.333e-02 | tov=2.290e-02 | bc=7.029e-02 | surf=9.794e-05 | K=49.875 (50.1%) | γ=1.5025 (24.9%) | ρ_c=4.788e-04 (62.6%)
[  100] total=1.280e+00 | data=8.597e-02 | tov=1.486e-02 | bc=4.018e-02 | surf=3.266e-03 | K=49.875 (50.1%) | γ=1.5025 (24.9%) | ρ_c=4.788e-04 (62.6%)
[  150] total=9.924e-01 | data=6.890e-02 | tov=1.990e-02 | bc=2.710e-02 | surf=1.251e-02 | K=49.875 (50.1%) | γ=1.5025 (24.9%) | ρ_c=4.788e-04 (62.6%)
[  200] total=8.195e-01 | data=6.582e-02 | tov=1.185e-02 | bc=1.481e-02 | surf=1.293e-03 | K=49.875 (50.1%) | γ=1.5025 (24.9%) | ρ_c=4.788e-04 (62.6%)
[  250] total=6.146e-01 | data=4.700e-02 | tov=1.894e-02 | bc=1.154e-02 | surf=1.028e-02 | K=49.875 (50.1%) | γ=1.5025 (24.9%) | ρ_c=4.788e-04 (62.6%)
[  300] total=5.503e-01 | data=4.665e-02 | tov=1.368e-02 | bc=6.687e-03 | surf=3.265e-03 | K=4